# Lab Task 02 — Effect of Image Filtering on Skin-Lesion Classification

**Dataset:** HAM10000 (Human Against Machine with 10000 training images)

**Best 3 models from Lab Activity 1 (Task 01):**
- **Model 1: VGG16** — 81.82% Accuracy, 97.32% AUC (best overall)
- **Model 2: ResNet18** — 81.82% Accuracy, 89.73% AUC
- **Model 3: AlexNet** — 72.73% Accuracy, 96.28% AUC (best AUC among the remaining tier)

**What this notebook does, section by section:**
1. Dataset Preparation
2. Image Filtering (Average, Gaussian, Median, Sharpening, Sobel)
3. Model Loading
4. Training & Evaluation Utilities
5. Baseline Experiment (No Filter)
6. Filtering Experiments (5 filters × 3 models)
7. Visualization
8. Comparative Analysis (final table answering the lab's required results)
9. Answers to the 10 lab questions
10. GitHub submission instructions

⚠️ **This trains 3 models × 6 conditions (baseline + 5 filters) = 18 runs.** To keep this realistic on Colab's free GPU, `EPOCHS` and `MAX_IMAGES_PER_CLASS` are kept small by default — raise them once you've confirmed everything runs, for your final submission numbers.

Set the runtime to GPU first: `Runtime > Change runtime type > T4 GPU`.

In [ ]:
!pip install -q kagglehub opencv-python-headless scikit-learn seaborn

## Section 1: Dataset Preparation

Downloads HAM10000 directly into Colab via `kagglehub` — no `kaggle.json` file needed. First run will prompt you to paste your Kaggle username + API key (from kaggle.com/settings → API → Create New Token — just copy-paste what it shows you).

In [ ]:
import kagglehub

path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
print("Dataset downloaded to:", path)

In [ ]:
import os

for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    if level < 2:
        indent = '  ' * level
        print(f"{indent}{os.path.basename(root) or root}/")
        for f in files[:3]:
            print(f"{indent}  {f}")

In [ ]:
import pandas as pd
import glob

# Load metadata
metadata_path = glob.glob(os.path.join(path, "**", "HAM10000_metadata.csv"), recursive=True)[0]
df = pd.read_csv(metadata_path)

# Map each image_id to its actual file path (HAM10000 images are split across two folders)
image_paths = {}
for f in glob.glob(os.path.join(path, "**", "*.jpg"), recursive=True):
    image_id = os.path.splitext(os.path.basename(f))[0]
    image_paths[image_id] = f

df["path"] = df["image_id"].map(image_paths)
df = df.dropna(subset=["path"]).reset_index(drop=True)

print("Total labeled images found:", len(df))
print("\nClass distribution (dx column):")
print(df["dx"].value_counts())

In [ ]:
import matplotlib.pyplot as plt

# HAM10000 class distribution
plt.figure(figsize=(8, 5))
df["dx"].value_counts().plot(kind="bar", color="steelblue")
plt.title("HAM10000 Class Distribution (all 7 classes)")
plt.xlabel("Diagnosis (dx)")
plt.ylabel("Number of images")
plt.tight_layout()
plt.show()

In [ ]:
# ---- EDIT THESE if you want different classes / dataset size ----
# HAM10000 classes: akiec, bcc, bkl, df, mel, nv, vasc
SELECTED_CLASSES = ["nv", "mel", "bkl", "bcc"]   # 4 classes, consistent with Lab Activity 1
MAX_IMAGES_PER_CLASS = 250                        # cap per class to keep 18 training runs feasible on free Colab

subset = df[df["dx"].isin(SELECTED_CLASSES)].copy()

balanced_parts = []
for cls in SELECTED_CLASSES:
    cls_df = subset[subset["dx"] == cls]
    n = min(len(cls_df), MAX_IMAGES_PER_CLASS)
    balanced_parts.append(cls_df.sample(n=n, random_state=42))
subset = pd.concat(balanced_parts).reset_index(drop=True)

class_to_idx = {cls: i for i, cls in enumerate(SELECTED_CLASSES)}
subset["label"] = subset["dx"].map(class_to_idx)

print("Selected classes:", SELECTED_CLASSES)
print("Class-to-index mapping:", class_to_idx)
print("\nFinal dataset size per class:")
print(subset["dx"].value_counts())
print("\nTotal images used:", len(subset))

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(subset, test_size=0.3, stratify=subset["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42)

print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")
# This exact split (same random_state, same proportions) is reused for every model/filter
# combination below, so all 18 experiments are compared fairly on identical data.

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image

class HAM10000Dataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, row["label"]

In [ ]:
import numpy as np

# Show a few original example images
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, cls in zip(axes, SELECTED_CLASSES):
    sample_path = subset[subset["dx"] == cls]["path"].iloc[0]
    img = Image.open(sample_path).convert("RGB")
    ax.imshow(img)
    ax.set_title(cls)
    ax.axis("off")
plt.suptitle("Original Image Examples (one per class)")
plt.tight_layout()
plt.show()

## Section 2: Image Filtering

Five spatial-domain filters, each implemented with OpenCV and wrapped as a torchvision-compatible transform.

In [ ]:
import cv2
import numpy as np

def apply_average(img_np):
    return cv2.blur(img_np, (5, 5))

def apply_gaussian(img_np):
    return cv2.GaussianBlur(img_np, (5, 5), 0)

def apply_median(img_np):
    return cv2.medianBlur(img_np, 5)

def apply_sharpen(img_np):
    kernel = np.array([[0, -1, 0],
                        [-1, 5, -1],
                        [0, -1, 0]])
    return cv2.filter2D(img_np, -1, kernel)

def apply_sobel(img_np):
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    magnitude = np.sqrt(sobel_x**2 + sobel_y**2)
    magnitude = np.uint8(255 * magnitude / (magnitude.max() + 1e-8))
    return cv2.cvtColor(magnitude, cv2.COLOR_GRAY2RGB)  # back to 3 channels so the same model input shape works

FILTER_FUNCTIONS = {
    "No Filter": None,
    "Average": apply_average,
    "Gaussian": apply_gaussian,
    "Median": apply_median,
    "Sharpening": apply_sharpen,
    "Sobel": apply_sobel,
}

class FilterTransform:
    """Applies one of the filter functions above to a PIL image, returns a PIL image."""
    def __init__(self, filter_fn):
        self.filter_fn = filter_fn

    def __call__(self, pil_img):
        if self.filter_fn is None:
            return pil_img
        img_np = np.array(pil_img)
        filtered_np = self.filter_fn(img_np)
        return Image.fromarray(filtered_np)

In [ ]:
# Visualize original vs every filter on one sample image
sample_path = subset["path"].iloc[0]
sample_img = Image.open(sample_path).convert("RGB")

fig, axes = plt.subplots(1, 6, figsize=(20, 4))
for ax, (name, fn) in zip(axes, FILTER_FUNCTIONS.items()):
    transformed = FilterTransform(fn)(sample_img)
    ax.imshow(transformed)
    ax.set_title(name)
    ax.axis("off")
plt.suptitle("Original vs Filtered Image Examples")
plt.tight_layout()
plt.show()

## Section 3: Model Loading (Best 3 Models from Lab Activity 1)

In [ ]:
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

NUM_CLASSES = len(SELECTED_CLASSES)

def build_model(name):
    if name == "AlexNet":
        m = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
        m.classifier[6] = nn.Linear(4096, NUM_CLASSES)
    elif name == "VGG16":
        m = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        m.classifier[6] = nn.Linear(4096, NUM_CLASSES)
    elif name == "ResNet18":
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    else:
        raise ValueError(f"Unknown model: {name}")
    return m.to(device)

MODEL_NAMES = ["VGG16", "ResNet18", "AlexNet"]   # best 3 from Lab Activity 1
FILTER_NAMES = ["No Filter", "Average", "Gaussian", "Median", "Sharpening", "Sobel"]
EPOCHS = 5   # keep small for 18 runs; raise for your final submission numbers

## Section 4: Training & Evaluation Utilities

In [ ]:
from torch.utils.data import DataLoader
from torchvision import transforms

BATCH_SIZE = 32
IMG_MEAN = [0.485, 0.456, 0.406]
IMG_STD = [0.229, 0.224, 0.225]

def get_dataloaders(filter_name):
    filter_fn = FILTER_FUNCTIONS[filter_name]
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        FilterTransform(filter_fn),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMG_MEAN, std=IMG_STD),
    ])
    train_loader = DataLoader(HAM10000Dataset(train_df, transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader   = DataLoader(HAM10000Dataset(val_df, transform),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader  = DataLoader(HAM10000Dataset(test_df, transform),  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    return train_loader, val_loader, test_loader

In [ ]:
def run_epoch(model, loader, criterion, optimizer, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.set_grad_enabled(train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

def train_model(model, train_loader, val_loader, epochs=EPOCHS):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for epoch in range(epochs):
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer, train=True)
        va_loss, va_acc = run_epoch(model, val_loader, criterion, optimizer, train=False)
        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        print(f"  Epoch {epoch+1}/{epochs}  train_acc={tr_acc:.3f}  val_acc={va_acc:.3f}")
    return model, history

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             balanced_accuracy_score, roc_auc_score, confusion_matrix, classification_report)
from sklearn.preprocessing import label_binarize

def evaluate_model(model, test_loader):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            preds = probs.argmax(1)
            all_labels.extend(labels.numpy())
            all_preds.extend(preds)
            all_probs.extend(probs)

    y_true, y_pred, y_prob = np.array(all_labels), np.array(all_preds), np.array(all_probs)

    acc = accuracy_score(y_true, y_pred) * 100
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0) * 100
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0) * 100
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0) * 100
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0) * 100
    bal_acc = balanced_accuracy_score(y_true, y_pred) * 100

    y_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
    try:
        auc = roc_auc_score(y_bin, y_prob, average='macro', multi_class='ovr') * 100
    except Exception:
        auc = float('nan')

    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred, target_names=SELECTED_CLASSES, output_dict=True, zero_division=0)

    metrics = {
        "Accuracy": acc, "Precision": prec, "Recall": rec, "F1-score": f1_weighted,
        "Macro-F1": macro_f1, "Balanced Accuracy": bal_acc, "AUC": auc,
    }
    return metrics, cm, report

## Section 5: Baseline Experiment (No Filter)

Run each of the 3 best models on the **original, unfiltered images** first — this is the baseline every filtered run gets compared against.

In [ ]:
all_results = []       # one row per (model, filter) combination -> final comparison table
all_histories = {}      # training curves, keyed by (model, filter)
all_confusion = {}      # confusion matrices, keyed by (model, filter)
all_reports = {}        # per-class precision/recall/F1, keyed by (model, filter)

def run_experiment(model_name, filter_name):
    print(f"\n{'='*60}\n{model_name} | {filter_name}\n{'='*60}")
    train_loader, val_loader, test_loader = get_dataloaders(filter_name)
    model = build_model(model_name)
    model, history = train_model(model, train_loader, val_loader)
    metrics, cm, report = evaluate_model(model, test_loader)

    print(f"  -> Accuracy={metrics['Accuracy']:.2f}%  Macro-F1={metrics['Macro-F1']:.2f}%  AUC={metrics['AUC']:.2f}%")

    row = {"Model": model_name, "Filter": filter_name, **{k + " (%)": round(v, 2) for k, v in metrics.items()}}
    all_results.append(row)
    all_histories[(model_name, filter_name)] = history
    all_confusion[(model_name, filter_name)] = cm
    all_reports[(model_name, filter_name)] = report

    del model
    torch.cuda.empty_cache() if device.type == "cuda" else None

In [ ]:
# Baseline: No Filter, for all 3 models
for model_name in MODEL_NAMES:
    run_experiment(model_name, "No Filter")

## Section 6: Filtering Experiments

Now run each of the 3 models with each of the 5 filters (15 more runs, 18 total including the baseline).

In [ ]:
for model_name in MODEL_NAMES:
    for filter_name in ["Average", "Gaussian", "Median", "Sharpening", "Sobel"]:
        run_experiment(model_name, filter_name)

print("\nAll 18 experiments complete.")

## Section 7: Visualization

In [ ]:
# Training/validation accuracy & loss curves - one figure per model, all 6 filter conditions overlaid
for model_name in MODEL_NAMES:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for filter_name in FILTER_NAMES:
        h = all_histories[(model_name, filter_name)]
        axes[0].plot(h["val_acc"], label=filter_name)
        axes[1].plot(h["val_loss"], label=filter_name)
    axes[0].set_title(f"{model_name} — Validation Accuracy")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy"); axes[0].legend()
    axes[1].set_title(f"{model_name} — Validation Loss")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss"); axes[1].legend()
    plt.tight_layout()
    plt.show()

In [ ]:
import seaborn as sns

# Confusion matrices - grid of 18 (3 models x 6 filters)
fig, axes = plt.subplots(len(MODEL_NAMES), len(FILTER_NAMES), figsize=(24, 12))
for i, model_name in enumerate(MODEL_NAMES):
    for j, filter_name in enumerate(FILTER_NAMES):
        cm = all_confusion[(model_name, filter_name)]
        ax = axes[i, j]
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                    xticklabels=SELECTED_CLASSES, yticklabels=SELECTED_CLASSES, ax=ax)
        ax.set_title(f"{model_name}\n{filter_name}", fontsize=9)
        if i == len(MODEL_NAMES) - 1:
            ax.set_xlabel("Predicted")
        if j == 0:
            ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
# Per-class precision/recall/F1 - example for the baseline of each model
for model_name in MODEL_NAMES:
    report = all_reports[(model_name, "No Filter")]
    report_df = pd.DataFrame(report).T.loc[SELECTED_CLASSES, ["precision", "recall", "f1-score"]] * 100
    print(f"\n{model_name} (No Filter) — per-class metrics (%):")
    print(report_df.round(2))

## Section 8: Comparative Analysis — Final Results Table

This is the table the lab task asks for: all 3 models × 6 conditions (baseline + 5 filters).

In [ ]:
results_df = pd.DataFrame(all_results)
results_df = results_df[["Model", "Filter", "Accuracy (%)", "Precision (%)", "Recall (%)",
                          "F1-score (%)", "Macro-F1 (%)", "AUC (%)"]]
results_df

In [ ]:
results_df.to_csv("lab02_results.csv", index=False)
print("Saved to lab02_results.csv")

In [ ]:
# Grouped bar chart: Accuracy by model and filter
pivot_acc = results_df.pivot(index="Filter", columns="Model", values="Accuracy (%)").loc[FILTER_NAMES]
pivot_acc.plot(kind="bar", figsize=(10, 6))
plt.title("Accuracy by Filter, per Model")
plt.ylabel("Accuracy (%)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

# Grouped bar chart: Macro-F1 by model and filter
pivot_f1 = results_df.pivot(index="Filter", columns="Model", values="Macro-F1 (%)").loc[FILTER_NAMES]
pivot_f1.plot(kind="bar", figsize=(10, 6))
plt.title("Macro-F1 by Filter, per Model")
plt.ylabel("Macro-F1 (%)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Change vs baseline (No Filter) for each model/filter/metric - answers "which filter changes results most"
delta_rows = []
for model_name in MODEL_NAMES:
    baseline = results_df[(results_df["Model"] == model_name) & (results_df["Filter"] == "No Filter")].iloc[0]
    for filter_name in ["Average", "Gaussian", "Median", "Sharpening", "Sobel"]:
        row = results_df[(results_df["Model"] == model_name) & (results_df["Filter"] == filter_name)].iloc[0]
        delta_rows.append({
            "Model": model_name,
            "Filter": filter_name,
            "Accuracy Delta (%)": round(row["Accuracy (%)"] - baseline["Accuracy (%)"], 2),
            "Macro-F1 Delta (%)": round(row["Macro-F1 (%)"] - baseline["Macro-F1 (%)"], 2),
            "AUC Delta (%)": round(row["AUC (%)"] - baseline["AUC (%)"], 2),
        })
delta_df = pd.DataFrame(delta_rows)
delta_df

## Section 9: Answers to the Lab Questions

Questions 1–6 are auto-answered below from your actual `results_df` / `delta_df`, so they'll update automatically when you re-run with different data or settings. Questions 7–10 are conceptual and answered in text based on established image-processing / deep-learning theory — read them and adjust in your own words for submission.

In [ ]:
# Q1: Which three pretrained models performed best in Lab Activity 1?
print("Q1: Best 3 models from Lab Activity 1 (by Table 1 Accuracy, AUC as tiebreaker):")
print("    1. VGG16 (81.82% Accuracy, 97.32% AUC)")
print("    2. ResNet18 (81.82% Accuracy, 89.73% AUC)")
print("    3. AlexNet (72.73% Accuracy, 96.28% AUC)")

In [ ]:
# Q2 & Q3: How does filtering affect each model, and which filter changes results the most?
print("Q2: Per-model effect of each filter (Accuracy Delta vs No Filter):\n")
print(delta_df.pivot(index="Filter", columns="Model", values="Accuracy Delta (%)").loc[["Average","Gaussian","Median","Sharpening","Sobel"]])

print("\nQ3: Filter with the largest absolute change vs baseline, per model:")
for model_name in MODEL_NAMES:
    sub = delta_df[delta_df["Model"] == model_name].copy()
    sub["abs_delta"] = sub["Accuracy Delta (%)"].abs()
    biggest = sub.loc[sub["abs_delta"].idxmax()]
    print(f"  {model_name}: {biggest['Filter']} ({biggest['Accuracy Delta (%)']:+.2f}% accuracy change)")

In [ ]:
# Q4: Does the effect of a filter remain consistent across all three models?
print("Q4: Direction of each filter's effect (positive/negative) across the 3 models:\n")
direction_table = delta_df.pivot(index="Filter", columns="Model", values="Accuracy Delta (%)").loc[["Average","Gaussian","Median","Sharpening","Sobel"]]
consistency = direction_table.apply(lambda row: "Consistent" if (row > 0).all() or (row < 0).all() else "Inconsistent", axis=1)
print(pd.concat([direction_table, consistency.rename("Consistency across models")], axis=1))

In [ ]:
# Q5: Does filtering improve or decrease macro-F1 and balanced accuracy on average?
print("Q5: Average Macro-F1 delta vs baseline, per filter (across all 3 models):")
print(delta_df.groupby("Filter")["Macro-F1 Delta (%)"].mean().loc[["Average","Gaussian","Median","Sharpening","Sobel"]].round(2))

In [ ]:
# Q6: Which lesion classes are most affected by filtering?
# Compares per-class F1 (No Filter) vs per-class F1 (each filter) for one model at a time
for model_name in MODEL_NAMES:
    print(f"\n--- {model_name}: per-class F1 change (Filter minus No Filter) ---")
    baseline_report = all_reports[(model_name, "No Filter")]
    baseline_f1 = {cls: baseline_report[cls]["f1-score"] * 100 for cls in SELECTED_CLASSES}
    for filter_name in ["Average", "Gaussian", "Median", "Sharpening", "Sobel"]:
        report = all_reports[(model_name, filter_name)]
        deltas = {cls: round(report[cls]["f1-score"] * 100 - baseline_f1[cls], 2) for cls in SELECTED_CLASSES}
        print(f"  {filter_name}: {deltas}")

**Q7. Why might smoothing remove useful lesion texture or morphological information?**

Average, Gaussian, and Median filters all work by averaging or ranking pixel values within a local neighborhood. Skin lesions are often distinguished by fine-grained texture (e.g. pigment network, dots/globules, irregular borders) — exactly the kind of high-frequency detail that smoothing filters suppress. By blurring these details, smoothing can erase the subtle morphological cues (border irregularity, texture heterogeneity) that a CNN relies on to distinguish, say, melanoma from a benign nevus.

**Q8. Why might sharpening or edge detection help or hurt classification?**

Sharpening amplifies high-frequency content (edges, texture), which can help emphasize lesion borders and surface texture — potentially useful since border irregularity is a clinically relevant feature. But it can also amplify noise and artifacts (hair, lighting variation), which may mislead the model. Sobel goes further and discards color/intensity information entirely, keeping only edge magnitude — this can help if the model's decision truly hinges on shape/border, but it destroys pigmentation and color cues (which are often diagnostically important in dermoscopy), so it can also hurt classification substantially.

**Q9. What is the difference between convolution and correlation?**

Both operations slide a kernel over an image and compute a weighted sum, but they differ in kernel orientation: **cross-correlation** slides the kernel as-is, while **convolution** first flips the kernel 180° (both horizontally and vertically) before sliding it. For a symmetric kernel (like a Gaussian or mean filter), the two are identical. For an asymmetric kernel (like a Sobel operator), they give different results. In practice, most deep learning frameworks (including PyTorch's `nn.Conv2d`) implement cross-correlation but call it "convolution," since the kernel weights are learned anyway and the flip makes no practical difference for a trained network.

**Q10. Based on your results, explain the relationship between classical image processing and deep-learning-based feature extraction.**

Classical filters and deep CNN feature extraction both operate on the same core idea — convolving a kernel across an image — but classical filters use fixed, hand-designed kernels (mean, Gaussian, Sobel, etc.) chosen to isolate a specific property (smoothness, edges), while a CNN's convolutional layers *learn* their own kernels from data to isolate whatever features minimize classification error. Applying a classical filter *before* the CNN effectively edits the input distribution the network was pretrained on (ImageNet natural images), which can help if the filter happens to emphasize a feature the network already relies on, or hurt if it destroys information the network needs and can't recover in later layers. The results above should show that filters are not universally good or bad — their effect depends on how well the "hand-crafted" transformation aligns with what the specific pretrained network already learned to look for.

## Section 10: Code Submission — Push to GitHub

Run these commands in a Colab code cell (or your local terminal) to push this notebook + a README to your GitHub repo under a `Lab 02` folder.

In [ ]:
# Save this notebook's results and prepare files for the GitHub folder
# (Run this in Colab; it assumes you've already downloaded this .ipynb into the Colab session,
#  or you can just drag this same .ipynb file into your repo folder manually instead.)

!mkdir -p "Lab 02"
!cp lab02_results.csv "Lab 02/"
print("Files ready in the 'Lab 02' folder. Add your .ipynb and README.md here too before pushing.")

In [ ]:
# From a terminal (or Colab cell prefixed with !), inside your cloned repo:

# git clone https://github.com/<your-username>/<your-repo>.git
# cd <your-repo>
# mkdir -p "Lab 02"
# cp /path/to/this/notebook.ipynb "Lab 02/"
# cp /path/to/README.md "Lab 02/"
# cp lab02_results.csv "Lab 02/"
# git add "Lab 02"
# git commit -m "Add Lab 02: Effect of Image Filtering on Skin-Lesion Classification"
# git push origin main